# Berry–Esseen: n25 B1 — 35 retained count forests

Choose a **CPU** runtime and **Runtime → Run all**. Authorize the same Google
Drive used by the other jobs. No parameter or path edits are required.
Run this queued job when a session becomes available; ongoing jobs need no changes.
Use one session per branch, with no concurrent copies of this same notebook.

The original complete candidates already exist. This batch freshly evaluates
all 47,946 selected leaves and runs all 24 original corruption controls for
each of the 35 forests. Historical PASS records are not acceptance inputs.
The mathematical source, contracts and candidates remain byte-for-byte unchanged.

Progress is displayed by forest and by the native supervisor. Logs are backed
up every minute. Each completed native acceptance is immediately saved as a
separate ZIP in `MyDrive/Berry-Esseen-end2end/n25-retained-b1/completed/`.
Rerunning this same notebook revalidates those complete stage ZIPs before
reusing them. An interrupted individual forest restarts its full acceptance;
it is never treated as a partly accepted proof.

Each forest has a 2 GiB process-group memory cap and a 12-hour native CPU ceiling.
These are finite safety budgets, not measured runtime estimates. The full batch
may span multiple sessions; keep the same Drive and the same pinned notebook.

Success verifies the selected conditional count arithmetic only. Global/local
resource acquisitions, analytic kernel soundness, the other n25 regions and
the all-index theorem remain separate obligations. No global PASS is claimed.


In [ ]:
from pathlib import Path
import subprocess, sys, json, os, time
CODE_COMMIT = "635da02141dd175c0f721cb0779c6fb433de6f8b"
TASK = "n25-retained-b1"
REPOSITORY = "https://github.com/rodziewiczk/berry-esseen-end2end.git"
CODE = Path("/content") / ("berry-esseen-code-" + CODE_COMMIT[:12])
print("Fetching the pinned source commit:", CODE_COMMIT, flush=True)
if not CODE.exists():
    subprocess.run(["git", "clone", "--quiet", "--no-checkout", REPOSITORY, str(CODE)], check=True)
    subprocess.run(["git", "-C", str(CODE), "checkout", "--quiet", "--detach", CODE_COMMIT], check=True)
observed = subprocess.check_output(["git", "-C", str(CODE), "rev-parse", "HEAD"], text=True).strip()
if observed != CODE_COMMIT:
    raise RuntimeError("Source commit mismatch")
subprocess.run(["git", "-C", str(CODE), "diff", "--exit-code", "HEAD", "--"], check=True)
# A rerun in an existing Colab kernel must not reuse modules from an older pin.
for module_name in tuple(sys.modules):
    if module_name == "be_replay" or module_name.startswith("be_replay."):
        del sys.modules[module_name]
sys.path.insert(0, str(CODE))
from be_replay.environment import snapshot
if not Path(sys.modules["be_replay"].__file__).resolve().is_relative_to(CODE.resolve()):
    raise RuntimeError("Imported execution code does not belong to the pinned checkout")
print(json.dumps(snapshot(), indent=2), flush=True)
print("Source code and environment recorded. Mathematical acceptance has not yet been established.", flush=True)


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DURABLE = Path("/content/drive/MyDrive/Berry-Esseen-end2end")
DURABLE.mkdir(parents=True, exist_ok=True)
print("Logs and checkpoints:", DURABLE, flush=True)


In [ ]:
from datetime import datetime, timezone
SELFTEST = DURABLE / ("selftest-" + TASK + "-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + ".log")
print("Testing the supervisor on the actual Linux runtime. These lightweight fixtures do not verify the mathematics.", flush=True)
with SELFTEST.open("w", encoding="utf-8") as log:
    child = subprocess.Popen([sys.executable, "-B", "-S", "-m", "unittest", "discover",
                              "-s", "tests", "-p", "test_linux_supervisor.py", "-v"],
                             cwd=CODE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in child.stdout:
        print(line, end="", flush=True)
        log.write(line); log.flush()
    code = child.wait()
if code != 0:
    raise RuntimeError("Supervisor self-tests failed; computation will not start. Log: " + str(SELFTEST))
print("Supervisor self-tests passed. Starting the selected calculation.", flush=True)


In [ ]:
print("Checking retained-count source, evidence and resume infrastructure. No mathematical kernel is executed by these fixtures.", flush=True)
DRIVER_SELFTEST = SELFTEST.with_name(SELFTEST.stem + "-retained.log")
with DRIVER_SELFTEST.open("w", encoding="utf-8") as log:
    result = subprocess.run([sys.executable, "-B", "-S", "-m", "unittest", "discover",
                             "-s", "tests", "-p", "test_retained_n25_*.py", "-v"],
                            cwd=CODE, stdout=log, stderr=subprocess.STDOUT, text=True)
lines = DRIVER_SELFTEST.read_text("utf-8").splitlines()
if result.returncode != 0:
    print("\n".join(lines), flush=True)
    raise RuntimeError("Retained-count infrastructure fixtures failed; computation will not start.")
print(next((line for line in reversed(lines) if line.startswith("Ran ")), "Fixtures passed."), flush=True)
print("Infrastructure fixture log:", DRIVER_SELFTEST, flush=True)


In [ ]:
from be_replay.retained_n25 import run_colab
RESULT = run_colab("b1", durable_root=DURABLE)


In [ ]:
from IPython.display import display, FileLink
from google.colab import files
print("Execution status:", RESULT["status"]["state"], flush=True)
print("The global proof remains unverified.", flush=True)
print("Results ZIP on Google Drive:", RESULT["bundle"], flush=True)
display(FileLink(RESULT["bundle"]))
try:
    files.download(RESULT["bundle"])
except Exception as error:
    print("Download the ZIP from the Google Drive path shown above:", error, flush=True)
